# Gymnasium: BipedalWalker-v3

Our objective is to train an agent to navigate the BipedalWalker environment using Reinforcement Learning. Before implementing complex algorithms or aiming for advanced maneuvers (like doing a flip), we need to understand the environment's dynamics.

## Environment Overview
`BipedalWalker-v3` is a 2D physics simulation environment from the Gymnasium Box2D environments. The goal is to make a bipedal robot walk to the right end of the terrain. In the `hardcore=True` version, the terrain is not flat; it includes obstacles such as ladders, stumps, and pitfalls.

### Action Space
The action space is a continuous `Box(-1.0, 1.0, (4,), float32)`. The agent controls the robot by applying torques to its four main joints. The four values in the action array represent:
1. Hip 1 (Torque / Speed)
2. Knee 1 (Torque / Speed)
3. Hip 2 (Torque / Speed)
4. Knee 2 (Torque / Speed)

All action values must be within the `[-1.0, 1.0]` range.

### Reward System
The agent receives rewards based on its forward progress and energy efficiency. According to the official documentation, the reward is calculated as follows:
* **Forward Movement:** The agent is rewarded for moving forward (to the right). Reaching the end of the terrain yields a total of over 300 points.
* **Falling Penalty:** If the robot's main body (hull) touches the ground, it falls. This results in a heavy penalty of **-100** points, and the episode terminates immediately.
* **Motor Torque Penalty:** To encourage efficient, natural walking rather than chaotic flailing, applying motor torque costs a small negative reward.
### Set up the Environment

In [1]:
import os
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

import gymnasium as gym
env = gym.make("BipedalWalker-v3", hardcore=False, render_mode="human") # Human we can see the environment

## Baseline: Random Actions

To establish a baseline and visualize how an untrained agent interacts with the physics engine, we will run a single episode using a random policy. The agent will sample actions uniformly from the action space until the episode ends.

An episode ends if:
- `terminated` is True (the agent falls or reaches the goal).
- `truncated` is True (the agent runs out of time/steps).
- Past 20 seconds of simulation time.

In [21]:
import time

env = gym.make("BipedalWalker-v3", hardcore=False, render_mode="human") 
limit_time = 10  # seconds
start_time = time.time()
obs, info = env.reset()

terminated = False
truncated = False
total_reward = 0.0
step_count = 0

# Loop until the agent finishes or fails
while not (terminated or truncated) and (time.time() - start_time < limit_time):
    # Sample a random continuous action within [-1, 1] for the 4 joints
    action = env.action_space.sample() 
    
    # Step the environment forward
    obs, reward, terminated, truncated, info = env.step(action)
    
    total_reward += reward
    step_count += 1

# Close the rendering window
env.close()

print(f"Episode finished after {step_count} steps.")
print(f"Total Reward with random policy: {total_reward:.2f}")

Episode finished after 52 steps.
Total Reward with random policy: -105.10


### Changing the Environment
Instead of training the agent only to walk, we can reshape the task so it learns to perform a flip.
The main idea is to change the reward and encourage trunk rotation, airtime, and landing control.


Changes in the robot:
- Track the robot body's orientation.
- Reward angular velocity and rotation progress.
- Give a large bonus when the agent completes a full rotation.
- Reduce the fall penalty so the agent is willing to take risks.
- Penalize forward movement less, so the policy focuses on flipping rather than walking.

In [ ]:
import gymnasium as gym
import numpy as np

class CurriculumFlipperWrapper(gym.Wrapper):
    def __init__(self, env, stage=1, max_steps=1500):
        super().__init__(env)
        self.max_steps = max_steps
        self.stage = stage # 1 for Rotation, 2 for Landing
        
        # Expand observation space by 1 to include cumulative_angle
        low = np.append(self.env.observation_space.low, -np.inf)
        high = np.append(self.env.observation_space.high, np.inf)
        self.observation_space = gym.spaces.Box(low, high, dtype=np.float32)

    def step(self, action):
        obs, reward, terminated, truncated, info = self.env.step(action)
        self.step_counter += 1

        current_angle = obs[0] # Hull angle
        delta_angle = current_angle - self.prev_angle

        # Handle wrap-around
        if delta_angle > np.pi:
            delta_angle -= 2 * np.pi
        elif delta_angle < -np.pi:
            delta_angle += 2 * np.pi

        prev_cumulative = self.cumulative_angle
        self.cumulative_angle += delta_angle
        self.prev_angle = current_angle

        custom_reward = 0.0

        # 1. Base Rotation Rewards (Active in both stages)
        angular_vel = obs[1]
        custom_reward += max(0.0, angular_vel) * 5.0 # Slightly reduced to prevent over-spinning
        
        rotation_progress = self.cumulative_angle - prev_cumulative
        custom_reward += max(0, rotation_progress) * 15.0

        # Airtime bonus
        if obs[8] == 0.0 and obs[13] == 0.0:
            custom_reward += 1.0

        # Intermediate milestones
        milestones = [np.pi / 2, np.pi, 3 * np.pi / 2]
        for milestone in milestones:
            attr = f"_milestone_{milestone:.2f}_done"
            if not getattr(self, attr, False) and self.cumulative_angle >= milestone:
                setattr(self, attr, True)
                custom_reward += 50.0

        # 2. Flip Completion (Do NOT terminate the episode here)
        if self.cumulative_angle >= 2 * np.pi and not self.flip_completed:
            self.flip_completed = True
            custom_reward += 300.0 # Reward the achievement

        # 3. Stage-Specific Logic
        is_falling = (reward == -100)
        
        if self.stage == 1:
            # Stage 1: Zero penalty for falling. We just want rotation.
            if is_falling:
                custom_reward += 100.0 # Neutralize the built-in -100 penalty
                
        elif self.stage == 2:
            # Stage 2: Landing constraints
            if self.flip_completed:
                # 1. Reward keeping the hull upright
                upright_bonus = np.cos(obs[0]) 
                custom_reward += max(0, upright_bonus) * 3.0 # Increased importance
                
                # 2. Penalize high rotational speed (force it to brake)
                custom_reward -= abs(angular_vel) * 3.0
                
                # 3. NEW: Penalize high downward velocity (force shock absorption)
                y_vel = obs[3] # Index 3 is linear velocity Y
                if y_vel < 0:
                    custom_reward += y_vel * 5.0 # y_vel is negative, so this reduces the reward
                
                # 4. Massive reward for planting both feet post-flip
                if obs[8] == 1.0 and obs[13] == 1.0:
                    custom_reward += 15.0 # Huge incentive to use legs
            
            # Keep the -100 fall penalty, but soften it slightly before the flip
            if is_falling and not self.flip_completed:
                custom_reward += 50.0

        # Time limit
        if self.step_counter >= self.max_steps:
            truncated = True

        info['flip_completed'] = self.flip_completed
        info['cumulative_angle'] = self.cumulative_angle

        # Re-apply the base reward (forward movement) + our custom shaped rewards
        total_reward = reward + custom_reward

        obs = np.append(obs, self.cumulative_angle / (2 * np.pi)).astype(np.float32)
        return obs, total_reward, terminated, truncated, info

    def reset(self, **kwargs):
        obs, info = self.env.reset(**kwargs)
        self.cumulative_angle = 0.0
        self.prev_angle = obs[0]
        self.flip_completed = False
        self.step_counter = 0
        for milestone in [np.pi / 2, np.pi, 3 * np.pi / 2]:
            setattr(self, f"_milestone_{milestone:.2f}_done", False)
        obs = np.append(obs, self.cumulative_angle / (2 * np.pi)).astype(np.float32)
        return obs, info

## Integrating Stable Baselines 3

Using Stable Baselines 3, we can implement a Proximal Policy Optimization (PPO) agent to learn how to flip in the BipedalWalker environment.

In [2]:
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import SubprocVecEnv

# --- STAGE 1: ROTATION ---
def make_env_stage1():
    def _init():
        e = gym.make("BipedalWalker-v3", hardcore=False)
        return CurriculumFlipperWrapper(e, stage=1)
    return _init

vec_env_s1 = SubprocVecEnv([make_env_stage1() for _ in range(32)])

model = PPO("MlpPolicy", vec_env_s1, verbose=1, device="cuda", 
            n_steps=2048, batch_size=256, learning_rate=3e-4,
            tensorboard_log="./ppo_bipedal_curriculum/")

print("Starting Stage 1: Rotation Mastery...")
model.learn(total_timesteps=5_000_000) 
model.save("ppo_bipedal_stage1")
vec_env_s1.close()

# --- STAGE 2: LANDING ---
def make_env_stage2():
    def _init():
        e = gym.make("BipedalWalker-v3", hardcore=False)
        return CurriculumFlipperWrapper(e, stage=2)
    return _init

vec_env_s2 = SubprocVecEnv([make_env_stage2() for _ in range(32)])

print("Starting Stage 2: Stabilization and Landing...")
# Load the weights from Stage 1 into the new environment
model = PPO.load("ppo_bipedal_stage1", env=vec_env_s2, device="cuda")

# Optional: Decay the learning rate for fine-tuning the landing
model.learning_rate = 1e-4 

model.learn(total_timesteps=2_000_000)
model.save("ppo_bipedal_final")
vec_env_s2.close()

Using cuda device
Starting Stage 1: Rotation Mastery...
Logging to ./ppo_bipedal_curriculum/PPO_1


c:\Users\gluca\miniconda3\envs\gymenv\Lib\site-packages\stable_baselines3\common\on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(


------------------------------
| time/              |       |
|    fps             | 4034  |
|    iterations      | 1     |
|    time_elapsed    | 16    |
|    total_timesteps | 65536 |
------------------------------
------------------------------------------
| time/                   |              |
|    fps                  | 2208         |
|    iterations           | 2            |
|    time_elapsed         | 59           |
|    total_timesteps      | 131072       |
| train/                  |              |
|    approx_kl            | 0.0045862575 |
|    clip_fraction        | 0.0366       |
|    clip_range           | 0.2          |
|    entropy_loss         | -5.65        |
|    explained_variance   | 0.00887      |
|    learning_rate        | 0.0003       |
|    loss                 | 3.78         |
|    n_updates            | 10           |
|    policy_gradient_loss | -0.0027      |
|    std                  | 0.992        |
|    value_loss           | 17.7         |
---------

### Testing Model


In [4]:
import time
import os
import gymnasium as gym
import torch
from stable_baselines3 import PPO

# 1. Corrected model path to match what was saved at the end of Stage 2
model_path = "ppo_bipedal_final.zip" 
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Create a single human-render environment and wrap it
env = gym.make("BipedalWalker-v3", hardcore=False, render_mode="human")

# 2. Corrected Wrapper name and explicitly set to Stage 2
env = CurriculumFlipperWrapper(env, stage=2)

if not os.path.exists(model_path):
    raise FileNotFoundError(f"Model not found: {model_path}. Train and save the model before running tests.")

print(f"Loading model from {model_path}...")
model = PPO.load(model_path, device=device)

num_episodes = 5
max_seconds = 15

for ep in range(1, num_episodes + 1):
    obs, info = env.reset()
    start_time = time.time()
    done = False
    total_reward = 0.0
    step_count = 0

    print(f"\n=== Test Episode {ep} ===")
    while not done and (time.time() - start_time) < max_seconds:
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated
        total_reward += reward
        step_count += 1

    flip_completed = info.get("flip_completed", False)
    final_angle = info.get("cumulative_angle", 0.0)

    print(f"Episode {ep} — steps: {step_count}, total_reward: {total_reward:.2f}")
    print(f"Flip completed: {flip_completed}, Final cumulative rotation: {final_angle:.2f} radians")

    time.sleep(0.5)

env.close()
print("All tests finished.")

Using device: cuda
Loading model from ppo_bipedal_final.zip...

=== Test Episode 1 ===
Episode 1 — steps: 43, total_reward: 166.89
Flip completed: False, Final cumulative rotation: 4.16 radians

=== Test Episode 2 ===
Episode 2 — steps: 43, total_reward: 166.78
Flip completed: False, Final cumulative rotation: 4.16 radians

=== Test Episode 3 ===
Episode 3 — steps: 44, total_reward: 169.88
Flip completed: False, Final cumulative rotation: 4.19 radians

=== Test Episode 4 ===
Episode 4 — steps: 43, total_reward: 167.82
Flip completed: False, Final cumulative rotation: 4.16 radians

=== Test Episode 5 ===
Episode 5 — steps: 43, total_reward: 167.83
Flip completed: False, Final cumulative rotation: 4.16 radians
All tests finished.
